# Visualization With Seaborn — with `xy.pyplot`

Code cells adapted from the [Python Data Science Handbook](https://github.com/jakevdp/PythonDataScienceHandbook) by Jake VanderPlas (MIT-licensed code; the book's prose is omitted).
The only systematic change is the import: `matplotlib.pyplot` → `xy.pyplot`, plus the same style-name/API modernizations the originals need to run on matplotlib ≥ 3.9 today.


# Visualization with Seaborn

In [ ]:
import xy.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

sns.set()  # seaborn's method to set its chart style

## Exploring Seaborn Plots

### Histograms, KDE, and Densities

In [ ]:
data = np.random.multivariate_normal([0, 0], [[5, 2], [2, 2]], size=2000)
data = pd.DataFrame(data, columns=["x", "y"])

for col in "xy":
    plt.hist(data[col], density=True, alpha=0.5)

In [ ]:
sns.kdeplot(data=data, shade=True);

In [ ]:
sns.kdeplot(data=data, x="x", y="y");

### Pair Plots

In [ ]:
iris = sns.load_dataset("iris")
iris.head()

In [ ]:
sns.pairplot(iris, hue="species", height=2.5);

### Faceted Histograms

In [ ]:
tips = sns.load_dataset("tips")
tips.head()

In [ ]:
tips["tip_pct"] = 100 * tips["tip"] / tips["total_bill"]

grid = sns.FacetGrid(tips, row="sex", col="time", margin_titles=True)
grid.map(plt.hist, "tip_pct", bins=np.linspace(0, 40, 15));

### Categorical Plots

In [ ]:
with sns.axes_style(style="ticks"):
    g = sns.catplot(x="day", y="total_bill", hue="sex", data=tips, kind="box")
    g.set_axis_labels("Day", "Total Bill")

### Joint Distributions

In [ ]:
with sns.axes_style("white"):
    sns.jointplot(x="total_bill", y="tip", data=tips, kind="hex")

In [ ]:
sns.jointplot(x="total_bill", y="tip", data=tips, kind="reg");

### Bar Plots

In [ ]:
planets = sns.load_dataset("planets")
planets.head()

In [ ]:
with sns.axes_style("white"):
    g = sns.catplot(x="year", data=planets, aspect=2, kind="count", color="steelblue")
    g.set_xticklabels(step=5)

In [ ]:
with sns.axes_style("white"):
    g = sns.catplot(
        x="year", data=planets, aspect=4.0, kind="count", hue="method", order=range(2001, 2015)
    )
    g.set_ylabels("Number of Planets Discovered")

## Example: Exploring Marathon Finishing Times

In [ ]:
# url = ('https://raw.githubusercontent.com/jakevdp/'
#        'marathon-data/master/marathon-data.csv')
# !cd data && curl -O {url}

In [ ]:
data = pd.read_csv("data/marathon-data.csv")
data.head()

In [ ]:
data.dtypes

In [ ]:
import datetime


def convert_time(s):
    h, m, s = map(int, s.split(":"))
    return datetime.timedelta(hours=h, minutes=m, seconds=s)


data = pd.read_csv(
    "data/marathon-data.csv", converters={"split": convert_time, "final": convert_time}
)
data.head()

In [ ]:
data.dtypes

In [ ]:
data["split_sec"] = data["split"].astype("int64") / 1e9
data["final_sec"] = data["final"].astype("int64") / 1e9
data.head()

In [ ]:
with sns.axes_style("white"):
    g = sns.jointplot(x="split_sec", y="final_sec", data=data, kind="hex")
    g.ax_joint.plot(np.linspace(4000, 16000), np.linspace(8000, 32000), ":k")

In [ ]:
data["split_frac"] = 1 - 2 * data["split_sec"] / data["final_sec"]
data.head()

In [ ]:
sns.displot(data["split_frac"], kde=False)
plt.axvline(0, color="k", linestyle="--");

In [ ]:
sum(data.split_frac < 0)

In [ ]:
g = sns.PairGrid(
    data, vars=["age", "split_sec", "final_sec", "split_frac"], hue="gender", palette="RdBu_r"
)
g.map(plt.scatter, alpha=0.8)
g.add_legend();

In [ ]:
sns.kdeplot(data.split_frac[data.gender == "M"], label="men", shade=True)
sns.kdeplot(data.split_frac[data.gender == "W"], label="women", shade=True)
plt.xlabel("split_frac");

In [ ]:
sns.violinplot(x="gender", y="split_frac", data=data, palette=["lightblue", "lightpink"]);

In [ ]:
data["age_dec"] = data.age.map(lambda age: 10 * (age // 10))
data.head()

In [ ]:
men = data.gender == "M"
women = data.gender == "W"

with sns.axes_style(style=None):
    sns.violinplot(
        x="age_dec",
        y="split_frac",
        hue="gender",
        data=data,
        split=True,
        inner="quartile",
        palette=["lightblue", "lightpink"],
    )

In [ ]:
(data.age > 80).sum()

In [ ]:
g = sns.lmplot(
    x="final_sec", y="split_frac", col="gender", data=data, markers=".", scatter_kws=dict(color="c")
)
g.map(plt.axhline, y=0.0, color="k", ls=":");